# Importing modules and settings

### Importing Libraries

In [ ]:
import numpy as np
import pandas  as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import os

In [ ]:
import gzip
import fnmatch
import re
from scipy.sparse import csr_matrix
import anndata as ad

### General settings of Scanpy

In [ ]:
sc.settings.figdir = './figures_250523/'

In [ ]:
sc.settings.verbosity = 4
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
# Create a CMAP for the UMAP plotting
umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:indigo'], as_cmap = True)

In [ ]:
# Declare the output file
name_of_analysis = 'Smed_L78-L47_20250523'
results_file = name_of_analysis + '_Results.h5ad'
results_file

# Loading the matrices, converting and annotating them

### L78, RNAi library

In [ ]:
with gzip.open('../matrix/L78/L78_1_G50_MQ0_matrix.txt.gz', 'rt') as f:
    data=pd.read_table(f, sep="\t")
counts=data.iloc[:,1:].values.T
genes=data.iloc[:,0].tolist()
cells=['L78_1_' + i for i in data.iloc[:,1:].columns.tolist()]
matrix=csr_matrix(counts, dtype=np.float32)
lib_78_1 = ad.AnnData(matrix)
lib_78_1.obs_names = cells
lib_78_1.var_names = genes
lib_78_1.obs["Experiment"]= 'RNAi'
lib_78_1.obs["Sublibrary"]= 'lib_78_1'

In [ ]:
with gzip.open('../matrix/L78/L78_2_G50_MQ0_matrix.txt.gz', 'rt') as f:
    data=pd.read_table(f, sep="\t")
counts=data.iloc[:,1:].values.T
genes=data.iloc[:,0].tolist()
cells=['L78_2_' + i for i in data.iloc[:,1:].columns.tolist()]
matrix=csr_matrix(counts, dtype=np.float32)
lib_78_2 = ad.AnnData(matrix)
lib_78_2.obs_names = cells
lib_78_2.var_names = genes
lib_78_2.obs["Experiment"]= 'RNAi'
lib_78_2.obs["Sublibrary"]= 'lib_78_2'

In [ ]:
with gzip.open('../matrix/L78/L78_3_G50_MQ0_matrix.txt.gz', 'rt') as f:
    data=pd.read_table(f, sep="\t")
counts=data.iloc[:,1:].values.T
genes=data.iloc[:,0].tolist()
cells=['L78_3_' + i for i in data.iloc[:,1:].columns.tolist()]
matrix=csr_matrix(counts, dtype=np.float32)
lib_78_3 = ad.AnnData(matrix)
lib_78_3.obs_names = cells
lib_78_3.var_names = genes
lib_78_3.obs["Experiment"]= 'RNAi'
lib_78_3.obs["Sublibrary"]= 'lib_78_3'

In [ ]:
with gzip.open('../matrix/L78/L78_4_G50_MQ0_matrix.txt.gz', 'rt') as f:
    data=pd.read_table(f, sep="\t")
counts=data.iloc[:,1:].values.T
genes=data.iloc[:,0].tolist()
cells=['L78_4_' + i for i in data.iloc[:,1:].columns.tolist()]
matrix=csr_matrix(counts, dtype=np.float32)
lib_78_4 = ad.AnnData(matrix)
lib_78_4.obs_names = cells
lib_78_4.var_names = genes
lib_78_4.obs["Experiment"]= 'RNAi'
lib_78_4.obs["Sublibrary"]= 'lib_78_4'

In [ ]:
with gzip.open('../matrix/L78/L78_5_G50_MQ0_matrix.txt.gz', 'rt') as f:
    data=pd.read_table(f, sep="\t")
counts=data.iloc[:,1:].values.T
genes=data.iloc[:,0].tolist()
cells=['L78_5_' + i for i in data.iloc[:,1:].columns.tolist()]
matrix=csr_matrix(counts, dtype=np.float32)
lib_78_5 = ad.AnnData(matrix)
lib_78_5.obs_names = cells
lib_78_5.var_names = genes
lib_78_5.obs["Experiment"]= 'RNAi'
lib_78_5.obs["Sublibrary"]= 'lib_78_5'

In [ ]:
with gzip.open('../matrix/L78/L78_6_G50_MQ0_matrix.txt.gz', 'rt') as f:
    data=pd.read_table(f, sep="\t")
counts=data.iloc[:,1:].values.T
genes=data.iloc[:,0].tolist()
cells=['L78_6_' + i for i in data.iloc[:,1:].columns.tolist()]
matrix=csr_matrix(counts, dtype=np.float32)
lib_78_6 = ad.AnnData(matrix)
lib_78_6.obs_names = cells
lib_78_6.var_names = genes
lib_78_6.obs["Experiment"]= 'RNAi'
lib_78_6.obs["Sublibrary"]= 'lib_78_6'

In [ ]:
adata = lib_78_1.concatenate(lib_78_2, lib_78_3, lib_78_4, lib_78_5, lib_78_6, join='outer', index_unique = None)
adata.obs = adata.obs.drop('batch', axis =1)

In [ ]:
adata.var_names_make_unique()
adata.var_names.astype(str)

In [ ]:
adata.var_names

In [ ]:
adata.var 

In [ ]:
adata.obs

In [ ]:
adata

#### Annotating the sample barcodes for L78

In [ ]:
barcode_path = 'L78_barcodes.xlsx'
barcodes = pd.read_excel(barcode_path, header = None, index_col = 0) 

In [ ]:
samp = 'Sample'
barcodes.rename( columns = {1:'bc', 2: samp}, inplace = True)

In [ ]:
barcodes.head(20)

In [ ]:
barcodes[samp] = barcodes[samp].astype('category')

In [ ]:
# Transfer the sample names into the adata object
for i in barcodes[samp].cat.categories: # Loop over sample column
    filt = (barcodes[samp] == i) # Filter the barcodes by sample name
    li = barcodes[filt]['bc'].to_list() # Add the filtered values to list
    for bc in li: # Loop over the barcodes list
        cellfilt = adata.obs.index.str.contains("_"+bc) # Filter by the barcode preceeded by "_"
        adata.obs.loc[cellfilt, samp] = i # Created new column with the sample names

In [ ]:
adata.obs

In [ ]:
check = 'CCGAGAATCC'
cfilt = adata.obs.index.str.contains("_"+check)

In [ ]:
adata.obs[cfilt]

In [ ]:
adata.obs[samp]

In [ ]:
adata.obs[samp] = adata.obs[samp].astype('category')

In [ ]:
adata.obs

In [ ]:
adata.obs[samp].cat.categories

In [ ]:
# Reorder by biological significance 
adata.obs[samp] = adata.obs[samp].cat.reorder_categories(['Cdh1_1', 'Cdh1_2', 'GFP_1', 'GFP_2', 'H2B_1', 'H2B_2'])

In [ ]:
# Create a count plot to show the counts of observations in each category 
sns.countplot(data=adata.obs, x=samp, palette='pastel')

#### Annotating biological and / or technical replicates.  

In [ ]:
# Extract the last part of the sample names (e.g., '1', '2') and store them in a new column named 'Replicates'
adata.obs['Replicate'] = adata.obs[samp].str.split('_').str[-1] 

In [ ]:
# Extract the first part of the sample names (e.g., '1', '2') and store them in a new column named 'condition'
adata.obs['Condition'] = adata.obs[samp].str.split('_').str[0] 

In [ ]:
adata.obs

### L47, FACS library

In [ ]:
with gzip.open('../matrix/L47/L47_1_G50_MQ0_matrix.txt.gz', 'rt') as f:
    data=pd.read_table(f, sep="\t")
counts=data.iloc[:,1:].values.T
genes=data.iloc[:,0].tolist()
cells=['L47_1_' + i for i in data.iloc[:,1:].columns.tolist()]
matrix=csr_matrix(counts, dtype=np.float32)
lib_47_1 = ad.AnnData(matrix)
lib_47_1.obs_names = cells
lib_47_1.var_names = genes
lib_47_1.obs["Experiment"]= 'FACS'
#lib_47_1.obs["Library"]= 'lib_47'
lib_47_1.obs["Sublibrary"]= 'lib_47_1'

In [ ]:
with gzip.open('../matrix/L47/L47_2_G50_MQ0_matrix.txt.gz', 'rt') as f:
    data=pd.read_table(f, sep="\t")
counts=data.iloc[:,1:].values.T
genes=data.iloc[:,0].tolist()
cells=['L47_2_' + i for i in data.iloc[:,1:].columns.tolist()]
matrix=csr_matrix(counts, dtype=np.float32)
lib_47_2 = ad.AnnData(matrix)
lib_47_2.obs_names = cells
lib_47_2.var_names = genes
lib_47_2.obs["Experiment"]= 'FACS'
#lib_47_2.obs["Library"]= 'lib_47'
lib_47_2.obs["Sublibrary"]= 'lib_47_2'

In [ ]:
with gzip.open('../matrix/L47/L47_3_G50_MQ0_matrix.txt.gz', 'rt') as f:
    data=pd.read_table(f, sep="\t")
counts=data.iloc[:,1:].values.T
genes=data.iloc[:,0].tolist()
cells=['L47_3_' + i for i in data.iloc[:,1:].columns.tolist()]
matrix=csr_matrix(counts, dtype=np.float32)
lib_47_3 = ad.AnnData(matrix)
lib_47_3.obs_names = cells
lib_47_3.var_names = genes
lib_47_3.obs["Experiment"]= 'FACS'
#lib_47_3.obs["Library"]= 'lib_47'
lib_47_3.obs["Sublibrary"]= 'lib_47_3'

In [ ]:
with gzip.open('../matrix/L47/L47_4_G50_MQ0_matrix.txt.gz', 'rt') as f:
    data=pd.read_table(f, sep="\t")
counts=data.iloc[:,1:].values.T
genes=data.iloc[:,0].tolist()
cells=['L47_4_' + i for i in data.iloc[:,1:].columns.tolist()]
matrix=csr_matrix(counts, dtype=np.float32)
lib_47_4 = ad.AnnData(matrix)
lib_47_4.obs_names = cells
lib_47_4.var_names = genes
lib_47_4.obs["Experiment"]= 'FACS'
#lib_47_4.obs["Library"]= 'lib_47'
lib_47_4.obs["Sublibrary"]= 'lib_47_4'

In [ ]:
adata_L47 = lib_47_1.concatenate(lib_47_2, lib_47_3, lib_47_4, join='outer', index_unique = None)
adata_L47.obs = adata_L47.obs.drop('batch', axis =1)

#### Annotating the sample barcodes for L47

All cells come from the same sample, the sublibraries were obtaines by FACS sorting. Sorting was done based on DNA content (DRAQ5 staining). L47_1 and L47_2 contain the 2c fraction (enriched in G1), L47_3 contains the 4c fraction (enriched in G2), L47_4 has both 2c and 4c

In [ ]:
libraries = list(set([val.split('_')[0] + '_' + val.split('_')[1] for val in adata_L47.obs.index]))
libraries.sort()
libraries

In [ ]:
cycle = ['G1a', 'G1b', 'G2', 'G1-G2']

In [ ]:
# assign the name to the sublibraries
for i, lib in enumerate(libraries):
    libfilt = adata_L47.obs.index.str.contains(lib)
    adata_L47.obs.loc[libfilt, 'Sample'] = cycle[i]

In [ ]:
sns.countplot(data=adata_L47.obs, x='Sample', palette='pastel') 

In [ ]:
cycle_type = ['G1', 'G1', 'G2', 'G1-G2']

In [ ]:
# name without replicate information
for i, lib in enumerate(libraries):
    libfilt = adata_L47.obs.index.str.contains(lib)
    adata_L47.obs.loc[libfilt, 'Condition'] = cycle_type[i]

In [ ]:
# replicate information
li_rep = []
for i in adata_L47.obs[samp]:
    li_rep.append(i[-1:])
li_rep2 = [x.replace('a', '1').replace('2', '1').replace('b', '2') for x in li_rep]
adata_L47.obs['Replicate'] = li_rep2

In [ ]:
adata_L47.obs

### Concatenate L78 and L47

In [ ]:
adata

In [ ]:
adata_L47

In [ ]:
adata = adata.concatenate(adata_L47, join = 'outer', index_unique = None)
adata.obs = adata.obs.drop('batch', axis =1)

In [ ]:
adata

In [ ]:
adata.obs

## Annotating library information.   

In [ ]:
# List the libraries 
libraries = list(set([val.split('_')[0] + '_' + val.split('_')[1] for val in adata.obs.index]))
libraries.sort() # Sort alphabetically
libraries

In [ ]:
# Loop over the different libraries and add a new column with the library name
for i, lib in enumerate(libraries):
    libfilt = adata.obs.index.str.contains(lib)
    adata.obs.loc[libfilt, 'Library'] = "L"+str(i+1)

In [ ]:
adata.obs

In [ ]:
adata.obs['Library'] = adata.obs['Library'].astype('category')

In [ ]:
adata.obs['Library'] 

In [ ]:
# Bar plot
sns.countplot(data=adata.obs, x='Library', palette = 'pastel')

In [ ]:
# Create a new column by combining the 'samp' and 'library columns'
adata.obs['Sample_Library'] = pd.Series(adata.obs[samp].astype('string') + "_" + adata.obs['Library'].astype('string'), dtype = object).astype('category')

In [ ]:
adata.obs

## Set the colors for the categories

In [ ]:
adata.obs['Condition'] = adata.obs['Condition'].astype('category')

In [ ]:
adata.obs['Condition'] = adata.obs['Condition'].cat.reorder_categories(['G1-G2', 'G1', 'G2', 'GFP', 'H2B', 'Cdh1'])

In [ ]:
adata.uns['Condition_colors'] = ['darkorange', 'darkred', 'gold' , 'slategray' , 'darkviolet', 'dodgerblue' ]

In [ ]:
adata.obs[samp] = adata.obs[samp].astype('category')

In [ ]:
adata.obs[samp] = adata.obs[samp].cat.reorder_categories(['G1-G2', 'G1a', 'G1b', 'G2', 'GFP_1', 'GFP_2',
       'H2B_1', 'H2B_2', 'Cdh1_1', 'Cdh1_2'])

In [ ]:
adata.uns['Sample_colors'] = [ 'darkorange',       # G1-G2
                                'darkred', 'lightcoral',    # G1 
                                 'gold',      # G2
                               'slategray', 'darkgray',   # GFP 
                                'darkviolet', 'mediumorchid',   # H2B 
                                'dodgerblue', 'skyblue'         # Cdh1 
                                ]

In [ ]:
adata.uns['Experiment_colors'] = ['darkred', 'dodgerblue']

# Generating pseudoreplicates 

In [ ]:
# Specify number of pseudoreplicates
how_many = 3

In [ ]:
# Create a list of pseudoreplicate labels, where each label is formed by concatenating the string "PS_" with a numerical suffix ranging from 1 to how_many.
pseudoreplicates = ["PS_" + str(i) for i in range(1,how_many+1)]
pseudoreplicates

In [ ]:
# Generate random values by randomly selecting pseudoreplicate labels from the pseudoreplicates list
random_values = np.random.choice(pseudoreplicates, size= len(adata.obs.index), replace=True)
random_values

In [ ]:
# Assign the array of randomly selected pseudoreplicate labels stored in the variable random_values to a new column named 'Pseudoreplicate'
adata.obs['Pseudoreplicate'] = random_values
adata.obs

In [ ]:
adata.obs['Pseudoreplicate'] = adata.obs['Pseudoreplicate'].astype('category')

In [ ]:
sns.countplot(data=adata.obs, x='Pseudoreplicate', palette = 'pastel')

In [ ]:
adata.obs

# Annotating the adata.var dataframe

In [ ]:
# Assign file path 
# this file is the same that was used in: 
# Pérez-Posada, A., García-Castro, H., Emili, E. et al. 
# Multimodal single cell analyses reveal gene networks of planarian stem cell differentiation. 
# Nat Commun 16, 10683 (2025). https://doi.org/10.1038/s41467-025-65712-0
# supplementary data 2
annot_path = "../annotation/rosetta_202311XX_with_wgcna_column.tsv" 


# Read the TSV file into a df
ann = pd.read_csv(annot_path, sep = '\t', index_col = 'gene') 

In [ ]:
ann

In [ ]:
# Retrieve the columns from the data frame and get a list of their names
ann.columns 

In [ ]:
# Retrieve number of columns
len(ann.columns) 

In [ ]:
# Return the variables of the adata object (the columns)
adata.var

In [ ]:
# Concatenate the adata.var df with df ann along the columns (axis 1) using an inner join.
adata.var = pd.concat([adata.var, ann], axis = 1, join = 'inner')

In [ ]:
adata.var

# doublet score with scrublet

In [ ]:
# Split the adata object into different batches based on 'Experiment'
batch_1 = adata[adata.obs['Experiment'] == 'RNAi']
batch_2 = adata[adata.obs['Experiment'] == 'FACS']

In [ ]:
# Apply Scrublet for Batch 1
sc.external.pp.scrublet(batch_1, batch_key='Experiment',
                        expected_doublet_rate=0.1, threshold=0.42)

# Apply Scrublet for Batch 2
sc.external.pp.scrublet(batch_2, batch_key='Experiment', 
                        expected_doublet_rate=0.1, threshold=0.31)

In [ ]:
adata.obs.loc[adata.obs['Experiment'] == 'RNAi', 'doublet_score'] = batch_1.obs['doublet_score']
adata.obs.loc[adata.obs['Experiment'] == 'RNAi', 'predicted_doublet'] = batch_1.obs['predicted_doublet']

adata.obs.loc[adata.obs['Experiment'] == 'FACS', 'doublet_score'] = batch_2.obs['doublet_score']
adata.obs.loc[adata.obs['Experiment'] == 'FACS', 'predicted_doublet'] = batch_2.obs['predicted_doublet']

In [ ]:
adata.obs

In [ ]:
# number of doublets
adata.obs['predicted_doublet'].value_counts()

In [ ]:
# number of doublets by sample
adata.obs[adata.obs['predicted_doublet'] == True].groupby(['Sample']).describe()

In [ ]:
doublets = adata.obs[adata.obs['predicted_doublet'] == True].groupby(['Sample']).describe()['doublet_score']['count']
allcells = adata.obs.groupby(['Sample']).describe()['doublet_score']['count']

In [ ]:
# percentage of doublets per sample
(doublets/allcells)*100

In [ ]:
# mean percentage of doublets
((doublets/allcells)*100).mean()

## save output

In [ ]:
# convert as str to avoid issues with saving the adata object
adata.obs['predicted_doublet']  = adata.obs['predicted_doublet'].astype('str')

In [ ]:
# save an h5ad file with the doublets
adata.write(name_of_analysis  + '_doublets.h5ad')